This notebook includes only the pipeline used for training the model. Please note that additional support will not be provided.



In [ ]:
# Importing necessary libraries and packages for the project

# Set the CUDA device for GPU computation (ensures reproducibility in multi-GPU settings)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = 'x'

# Core Python libraries for mathematical operations, statistics, and file handling
import math  # Provides mathematical functions
import statistics  # Provides functions for statistical calculations
import json  # To handle JSON data for input/output operations
import itertools  # Tools for iterating over combinations and permutations
import shutil  # For high-level file operations
import random  # Random number generation for data sampling and shuffling

# Scientific computing and data handling libraries
import numpy as np  # Numerical operations, arrays, and linear algebra
import pandas as pd  # Data manipulation and analysis using DataFrames

# Visualization libraries
import matplotlib.pyplot as plt  # Visualization of data, metrics, and results

# Progress bar visualization for loops and computations
from tqdm.auto import tqdm  # Provides a progress bar for loops

# Libraries for machine learning model development
from sklearn.model_selection import train_test_split, StratifiedKFold  # Data splitting and cross-validation
from sklearn.metrics import (
    roc_auc_score,  # ROC-AUC metric for classification performance
    f1_score,  # F1-score for classification evaluation
    matthews_corrcoef,  # Matthews Correlation Coefficient for binary classification
    average_precision_score  # Average precision metric for precision-recall curves
)
from sklearn.utils.class_weight import compute_class_weight  # Computes class weights for imbalanced datasets

# MONAI: A specialized framework for medical imaging AI
from monai.metrics import ROCAUCMetric, ConfusionMatrixMetric  # Evaluation metrics tailored for medical imaging
from monai.transforms import *  # Data augmentation, preprocessing, and transformation utilities

# Additional MONAI imports for deep learning and pipeline setup
import monai as mn  # Core MONAI library
from monai.config import print_config  # Prints MONAI configuration details
from monai.data import DataLoader, decollate_batch  # Data loaders and batch handling
from monai.transforms import (
    Activations, Activationsd, AsDiscrete, AsDiscreted, Compose,  # Activation and postprocessing transformations
    LoadImaged, EnsureTyped, EnsureChannelFirstd, ToTensord,  # Data loading and tensor conversion
    NormalizeIntensityd, Resized, Spacingd, Orientationd,  # Preprocessing steps: normalization and resizing
    RandAffineD, RandFlipd, RandSpatialCropd, RandScaleIntensityd, RandShiftIntensityd  # Data augmentation
)
from monai.utils import set_determinism  # Ensures reproducibility

# Libraries for handling medical imaging formats
import nibabel as nib  # For reading and writing NIfTI medical image files

# PyTorch: A machine learning library for building and training deep learning models
import torch  # Core PyTorch library
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # Dataset handling and samplers

# W&B: Weights & Biases for experiment tracking and logging
import wandb  # Tracks model performance and logs experiments

# Temporary file management, benchmarking, and timing tools
import tempfile  # Temporary file creation for caching data
import time  # Timing functions for performance benchmarking

# Printing MONAI configuration to check versions and dependencies
print_config()


MONAI version: 1.3.0

Numpy version: 1.26.2

Pytorch version: 2.1.2+cu121

MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 865972f7a791bf7b42efbcd87c8402bd865b329e
MONAI __file__: /home/<username>/anaconda3/envs/map_env/lib/python3.10/site-packages/monai/__init__.py

Other dependencies:

Pytorch Ignite version: 0.4.11

ITK version: 5.3.0

Nibabel version: 5.2.0

scikit-image version: 0.22.0

scipy version: 1.11.4

Pillow version: 10.2.0

Tensorboard version: 2.15.1

gdown version: 4.7.1

TorchVision version: 0.16.2+cu121

tqdm version: 4.66.1

lmdb version: 1.4.1

psutil version: 5.9.7

pandas version: 2.1.4

einops version: 0.7.0

transformers version: 4.36.2

mlflow version: 2.9.2

pynrrd version: 1.0.0

clearml version: 1.13.3rc0



In [ ]:
# Setting environment variables for Weights & Biases (W&B) API and logging behavior

# We set the W&B API key for authentication
# Replace this with your own API key to enable Weights & Biases experiment tracking
os.environ['WANDB_API_KEY'] = 'Your API Key code from your W&B'

# Suppressing W&B output logs to make the notebook output cleaner
os.environ['WANDB_SILENT'] = 'true'

# Function to set random seeds for reproducibility across all libraries and environments
def seed_all(seed: int) -> None:
    """
    Sets the seed for random number generators to ensure reproducibility across:
    - Python's `random` library
    - NumPy's random number generator
    - PyTorch (CPU and GPU)
    - MONAI for medical imaging workflows

    Args:
        seed (int): The seed value to set for all libraries.
    """
    # Set seed for Python's built-in random module
    random.seed(seed)

    # Set PYTHONHASHSEED environment variable for Python hash randomization
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Set seed for NumPy random number generator
    np.random.seed(seed)

    # Set seed for PyTorch's random number generator (CPU)
    torch.manual_seed(seed)

    # Set seed for PyTorch's random number generator (GPU)
    torch.cuda.manual_seed(seed)

    # Ensure deterministic behavior in PyTorch's CUDA backend
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False    # Disables auto-optimization for reproducibility

    # Set deterministic behavior for MONAI (for transformations, etc.)
    mn.utils.misc.set_determinism(seed=seed)

# Call the function to set the seed for reproducibility
seed_all(3366)  # We used 3366 as the fixed seed value

In [ ]:
# Experiment Configuration
# ==============================

# Experiment series: A name for the current experiment
experiment = "Choose_your _own_name"
# This name helps organize and identify experiments, particularly for logging and result tracking.

# ==============================
# Neural Network Architecture
# ==============================

# Specify the neural network architecture to be used
nn_architecture = "DenseNet121"
# for efficient feature propagation and gradient flow.

# ==============================
# Preprocessing Pipeline
# ==============================

# We define a preprocessing mode for data.
preprocessing_mode = 'FETS'
# 'FETS' refers to a specific preprocessing pipeline, from the Federated Tumor Segmentation (FETS) Challenge,
# ensuring standard input preprocessing for MRI data.

# ==============================
# MRI Sequence Selection
# ==============================

# Specify which MRI sequences to use as inputs
# Possible options: ['CT1_path', 'T2_path'] or combinations of these.
# Uncomment to choose the appropriate MRI sequence(s)

# mri_to_use = ['CT1_path','T2_path']  # Use both CT1 and T2 images
# mri_to_use = ['CT1_path']  # Use only CT1 images
# mri_to_use = ['T2_path']  # Use only T2 images

# Extract the MRI sequence names dynamically for tracking and file naming
mrisq = [mri.split('_')[0] for mri in mri_to_use]
# This line extracts the first part (e.g., "CT1" or "T2") from each string in `mri_to_use`.

# Join the selected MRI sequences into a single string for easier tracking
mri_sequences = '_'.join(mrisq)
# Example result: 'CT1' or 'CT1_T2'

# ==============================
# Results Directory
# ==============================

# Define the folder where experiment results will be saved
results_folder = 'the_path_to_the_director_that_you_want_to_save_your_models'
# Ensure that this directory path exists and has the appropriate write permissions.

In [ ]:
# ==============================
# Hyperparameter Configuration
# ==============================

#We added our selected hyperparameters.

# Batch size: Number of samples processed in one training iteration
bs = 16
# A smaller batch size like 16 is often used for medical imaging tasks, where models process high-resolution inputs
# and GPU memory can be a limiting factor.

# Learning rate: Step size for updating model parameters during training
lr = 1e-3
# A learning rate of 0.001 (1e-3) is a commonly used default. Adjust based on convergence behavior or fine-tuning needs.

# Device selection: Automatically use GPU if available, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# This line checks for GPU availability:
# - 'cuda': Use GPU for faster computation if CUDA is available.
# - 'cpu': Fallback to CPU if no GPU is available.

# Number of training epochs: Total number of passes through the entire training dataset
epochs = 650
# 650 epochs indicate a long training duration, which may be required for deep learning on medical imaging datasets.
# It allows the model to converge effectively, especially for complex data or small datasets.


**Data**

In [ ]:
# =====================================
# Training Data: All Data vs Cross-Validation
# =====================================

import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# =====================================
# Case 1: Training on All Data
# =====================================

# Load your full dataset into a pandas DataFrame
# Replace '<your_training_csv_path>' with the path to your training CSV file
df_train = pd.read_csv('<your_training_csv_path>')

# Convert the training DataFrame to a list of dictionaries (row-wise)
train_data_list = df_train.to_dict('records')

# Extract relevant columns for training
train_list = [
    {k: v for k, v in d.items() if k in ['subject_id_column_name', 'T2_path_column_name','CT1_path_column_name', 'label_column_name']}
    for d in train_data_list
]

# Instructions:
# - If you are training on all the data without validation, remove or ignore any validation loading code.
# - You can directly proceed with `train_list` for training.

# =====================================
# Case 2: Five-Fold Cross-Validation
# =====================================

# If you plan to perform five-fold cross-validation, use the following code to split and save the data.

# Load the full dataset
df = pd.read_csv('<your_full_csv_path>')  # Replace with your dataset path

# Set parameters for cross-validation
n_splits = 5  # Number of folds
seed = 6630  # Random seed for reproducibility

# Define the StratifiedKFold object
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

# Define the directory where the fold CSV files will be saved
output_dir = '<your_output_directory>'  # Replace with desired output path
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Perform Stratified K-Fold and save train/validation splits
for i, (train_index, val_index) in enumerate(skf.split(df, df['label_column_name'])):
    # Create train and validation DataFrames for this fold
    train_df = df.iloc[train_index]
    val_df = df.iloc[val_index]

    # Define file names for this fold
    train_filename = os.path.join(output_dir, f'train_fold_{i}.csv')
    val_filename = os.path.join(output_dir, f'val_fold_{i}.csv')

    # Save the train and validation splits to CSV
    train_df.to_csv(train_filename, index=False)
    val_df.to_csv(val_filename, index=False)

# =====================================
# Instructions:
# - Replace `<your_full_csv_path>` with the path to your complete dataset.
# - Replace `<your_output_directory>` with the folder where you want to save the CSV files.
# - If you are training on all data (no cross-validation), you don't need this code.
# - For five-fold cross-validation, this code generates 5 pairs of train and validation CSV files,
#   named `train_fold_0.csv`, `val_fold_0.csv`, ..., `train_fold_4.csv`, `val_fold_4.csv`.


In [ ]:
# =====================================
# Loading Training and Validation Data
# =====================================

import pandas as pd

# Load your training CSV file
# Replace '<your_training_csv_path>' with the path to your training dataset
df_train = pd.read_csv('<your_training_csv_path>')

# Convert the training DataFrame to a list of dictionaries (row-wise)
train_data_list = df_train.to_dict('records')

# If you are using validation data, uncomment and replace '<your_validation_csv_path>'
# with the path to your validation dataset
# df_val = pd.read_csv('<your_validation_csv_path>')
# val_data_list = df_val.to_dict('records')

# =====================================
# Extract Relevant Columns for Training
# =====================================

# Create a new list of dictionaries containing only the desired columns
train_list = [
    {k: v for k, v in d.items() if k in ['subject_id_column_name', 'T2_path_column_name','CT1_path_column_name', 'label_column_name']}
    for d in train_data_list
]

# If using validation data, extract relevant columns in a similar manner
# val_list = [
#     {k: v for k, v in d.items() if k in ['subject_id_column_name', 'T2_path_column_name','CT1_path_column_name', 'label_column_name']}  for d in val_data_list
# ]

# =====================================
# Instructions:
# - Replace '<your_training_csv_path>' with the path to your training CSV file.
# - If you are training on all available data (no validation split), you can ignore or remove
#   the validation CSV file loading and associated code (`val_data_list` and `val_list`).
# - Ensure your CSV file contains the necessary columns: 'subject_id_column_name', 'T2_path_column_name','CT1_path_column_name', 'label_column_name'.


In [ ]:
# =========================================
# Preprocessing and Data Augmentation
# =========================================

# Check preprocessing mode and apply corresponding transformations
if preprocessing_mode == 'FETS-std':
    print('Using transforms for data preprocessed using FETS')

    # ---------------------------------
    # Training Data Transformations
    # ---------------------------------
    train_transforms = mn.transforms.Compose([
        # 1. Load MRI images
        mn.transforms.LoadImageD(keys=mri_to_use),
        # Loads the image paths specified in 'mri_to_use' into memory.

        # 2. Ensure Channel First Format
        mn.transforms.EnsureChannelFirstd(keys=mri_to_use),
        # Converts images to channel-first format (C, H, W, D), as required by deep learning models.

        # 3. Normalize Intensity
        mn.transforms.NormalizeIntensityD(keys=mri_to_use, channel_wise=True),
        # Performs intensity normalization per channel to ensure consistent input ranges.
        # Helps stabilize training by centering pixel values.

        # 4. Concatenate MRI Sequences [If you are training only with T2 or CT1 you can skip this part]
        mn.transforms.ConcatItemsd(keys=mri_to_use, name='concat_mri'),
        # Combines multiple MRI sequences (e.g., 'CT1', 'T2') into a single tensor with multiple channels.

        # 5. Random Flipping for Data Augmentation
        mn.transforms.RandFlipd(keys='concat_mri', prob=0.5, spatial_axis=[0, 1, 2]),
        # Randomly flips the input image along spatial axes (x, y, z) with a 50% probability.

        # 6. Random Affine Transformations for Augmentation
        mn.transforms.RandAffineD(
            keys='concat_mri',
            translate_range=(15, 15, 10),  # Random translations in voxel space
            scale_range=(0.05, 0.05, 0.05),  # Random scaling with a ±5% range
            rotate_range=(math.pi / 8, math.pi / 8, math.pi / 8),  # Random rotations up to 22.5 degrees
            padding_mode='border',  # Border padding for transformations
            prob=0.5),  # 50% probability of applying the affine transform

        # 7. Convert to Tensor
        mn.transforms.ToTensord(keys=['concat_mri', 'label_column_name']),
        # Converts the input image and labels into PyTorch tensors for model input.
    ])

    # ---------------------------------
    # Validation Data Transformations
    # ---------------------------------
    val_transforms = mn.transforms.Compose([
        # 1. Load MRI images
        mn.transforms.LoadImageD(keys=mri_to_use),

        # 2. Ensure Channel First Format
        mn.transforms.EnsureChannelFirstd(keys=mri_to_use),

        # 3. Normalize Intensity
        mn.transforms.NormalizeIntensityD(keys=mri_to_use, channel_wise=True),

        # 4. Concatenate MRI Sequences
        mn.transforms.ConcatItemsd(keys=mri_to_use, name='concat_mri'),

        # 5. Convert to Tensor
        mn.transforms.ToTensord(keys=['concat_mri', 'label_column_name']),
    ])


In [ ]:
# =========================================
# Creating Training and Validation Datasets
# =========================================

# Create the training dataset
train_ds = mn.data.Dataset(data=train_list, transform=train_transforms)
# Explanation:
# - `data=train_list`: The training data, a list of dictionaries where each dictionary contains
#   the input MRI paths and corresponding labels.
# - `transform=train_transforms`: The preprocessing and augmentation pipeline to be applied to each sample
#   in the training dataset.

# Create the validation dataset
val_ds = mn.data.Dataset(data=val_list, transform=val_transforms)
# Explanation:
# - `data=val_list`: The validation data, structured similarly to the training data but used for evaluation.
# - `transform=val_transforms`: The preprocessing pipeline applied to validation data
#   (no augmentations, only standard transformations).


In [ ]:
# =========================================
# Inspecting the Shape of Processed Data
# =========================================

# Print the shape of the first sample's MRI tensor in the training and validation datasets
print(train_ds[0]['concat_mri'].shape, val_ds[0]['concat_mri'].shape)

# Explanation:
# - `train_ds[0]`: Retrieves the first sample from the training dataset.
# - `val_ds[0]`: Retrieves the first sample from the validation dataset.
# - `['concat_mri']`: Accesses the concatenated MRI tensor (created by `ConcatItemsd` transform).
# - `.shape`: Returns the dimensions of the MRI tensor, typically in the format (C, H, W, D):
#     - `C`: Number of channels (e.g., 1 for a single MRI sequence, or more for multiple sequences like CT1/T2).
#     - `H, W, D`: Height, width, and depth of the 3D MRI volume.

# Expected Output:
# - The printed shapes will help confirm that the transformations have been applied correctly.
# - Example output for single-channel MRIs:
#   torch.Size([1, 128, 128, 128]) torch.Size([1, 128, 128, 128])
#   Here:
#   - `1`: Single MRI channel (e.g., CT1).
#   - `128, 128, 128`: Spatial dimensions of the 3D volume after preprocessing.


**Plot some examples from the training and validation folds**

**Training fold**

In [ ]:
# =========================================
# Visualizing MRI Slices from the Dataset
# =========================================

# If both MRI sequences (e.g., CT1 and T2) are being used
if len(mri_to_use) == 2:
    # Select the subject and slice number for visualization
    train_subject_number = 0  # Index of the subject to visualize
    train_z_slice_img = 80  # Slice index to visualize along the z-axis
    train_subj_id = train_ds[train_subject_number]['subject_id_column_name']  # Subject ID
    train_sample_img = train_ds[train_subject_number]['concat_mri']  # Loaded and transformed MRI tensor

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the main title displaying the subject and slice information
    fig.suptitle(f"Slice {train_z_slice_img} from subject: {train_subj_id}", fontsize=15)

    # Add subplots to visualize each MRI sequence
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.imshow(train_sample_img[0, :, :, train_z_slice_img], cmap='gray')  # First channel (e.g., CT1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.imshow(train_sample_img[1, :, :, train_z_slice_img], cmap='gray')  # Second channel (e.g., T2)

    # Add titles for each subplot
    ax1.title.set_text("T1c")  # Title for the first channel
    ax2.title.set_text("T2")   # Title for the second channel

    plt.show()

# If only the 'T2' sequence is being used
elif "T2_path" in mri_to_use:
    # Select the subject and slice number for visualization
    train_subject_number = 0
    train_z_slice_img = 80
    train_subj_id = train_ds[train_subject_number]['subject_id_column_name']
    train_sample_img = train_ds[train_subject_number]['concat_mri']

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the main title displaying the subject and slice information
    fig.suptitle(f"Slice {train_z_slice_img} from subject: {train_subj_id}", fontsize=15)

    # Visualize the T2 image
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.imshow(train_sample_img[0, :, :, train_z_slice_img], cmap='gray')  # Single channel (T2)

    # Add title for the subplot
    ax2.title.set_text("T2")

    plt.show()

# If only the 'CT1' sequence is being used
elif "CT1_path" in mri_to_use:
    # Select the subject and slice number for visualization
    train_subject_number = 0
    train_z_slice_img = 80
    train_subj_id = train_ds[train_subject_number]['subject_id_column_name']
    train_sample_img = train_ds[train_subject_number]['concat_mri']

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the main title displaying the subject and slice information
    fig.suptitle(f"Slice {train_z_slice_img} from subject: {train_subj_id}", fontsize=15)

    # Visualize the CT1 image
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.imshow(train_sample_img[0, :, :, train_z_slice_img], cmap='gray')  # Single channel (CT1)

    # Add title for the subplot
    ax1.title.set_text("T1c")

    plt.show()


**Validation fold**

In [ ]:
# =========================================
# Visualizing MRI Slices from the Validation Dataset
# =========================================

# If both MRI sequences (e.g., CT1 and T2) are being used
if len(mri_to_use) == 2:
    # Select the subject and slice number for visualization
    val_subject_number = 6  # Index of the subject to visualize
    val_z_slice_img = 80  # Slice index to visualize along the z-axis
    val_subj_id = val_ds[val_subject_number]['subject_id_column_name']  # Subject ID
    val_sample_img = val_ds[val_subject_number]['concat_mri']  # Loaded and transformed MRI tensor

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the main title displaying the subject and slice information
    fig.suptitle(f"Slice {val_z_slice_img} from subject: {val_subj_id}", fontsize=15)

    # Add subplots to visualize each MRI sequence
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.imshow(val_sample_img[0, :, :, val_z_slice_img], cmap='gray')  # First channel (e.g., CT1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.imshow(val_sample_img[1, :, :, val_z_slice_img], cmap='gray')  # Second channel (e.g., T2)

    # Add titles for each subplot
    ax1.title.set_text("T1c")  # Title for the first channel
    ax2.title.set_text("T2")   # Title for the second channel

    plt.show()

# If only the 'T2' sequence is being used
elif "T2_path" in mri_to_use:
    # Select the subject and slice number for visualization
    val_subject_number = 6
    val_z_slice_img = 80
    val_subj_id = val_ds[val_subject_number]['subject_id_column_name']
    val_sample_img = val_ds[val_subject_number]['concat_mri']

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the main title displaying the subject and slice information
    fig.suptitle(f"Slice {val_z_slice_img} from subject: {val_subj_id}", fontsize=15)

    # Visualize the T2 image
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.imshow(val_sample_img[0, :, :, val_z_slice_img], cmap='gray')  # Single channel (T2)

    # Add title for the subplot
    ax2.title.set_text("T2")

    plt.show()

# If only the 'CT1' sequence is being used
elif "CT1_path" in mri_to_use:
    # Select the subject and slice number for visualization
    val_subject_number = 6
    val_z_slice_img = 80
    val_subj_id = val_ds[val_subject_number]['subject_id_column_name']
    val_sample_img = val_ds[val_subject_number]['concat_mri']

    # Plot the figure
    fig = plt.figure(figsize=(10, 10))

    # Adjust spacing between rows and main title
    fig.subplots_adjust(hspace=0.4, top=0.90)

    # Add the


**Model**

In [ ]:

# =========================================
# Creating Data Loaders for Training and Validation
# =========================================

# Training DataLoader
train_loader = DataLoader(
    train_ds,                     # Dataset: The training dataset
    batch_size=bs,                # Number of samples per batch
    num_workers=4,                # Number of subprocesses to use for data loading
    pin_memory=torch.cuda.is_available(),  # Enables fast data transfer to GPU memory if CUDA is available
    prefetch_factor=1,            # Number of samples preloaded by each worker into the buffer
    shuffle=True                  # Randomly shuffles the data at each epoch for better generalization
)

# Validation DataLoader
val_loader = DataLoader(
    val_ds,                       # Dataset: The validation dataset
    batch_size=bs,                # Number of samples per batch
    num_workers=4,                # Number of subprocesses to use for data loading
    pin_memory=torch.cuda.is_available(),  # Enables fast data transfer to GPU memory if CUDA is available
    prefetch_factor=1             # Number of samples preloaded by each worker into the buffer
    # Note: No shuffling for validation data to ensure consistent evaluation
)


In [ ]:
# =========================================
# Model, Loss Function, Optimizer, and Scheduler Setup
# =========================================

# 1. Model: Creating the DenseNet121 architecture
model = mn.networks.nets.DenseNet121(
    spatial_dims=3,               # Specifies the input data is 3D (e.g., MRI volumes)
    in_channels=len(mri_to_use),  # Number of input channels (e.g., 1 for CT1 or T2, 2 for both)
    out_channels=2                # Number of output classes (binary classification: e.g., 0 or 1)
).to(device)                      # Move the model to the appropriate device (GPU or CPU)

# Explanation:
# - DenseNet121: A pre-built dense convolutional neural network from MONAI.
# - `spatial_dims=3`: Handles volumetric medical imaging data (3D spatial inputs).
# - `in_channels`: Dynamically set based on the number of MRI sequences being used.
# - `out_channels=2`: Outputs logits for two classes; suitable for binary classification.

# 2. Loss Function: Cross-Entropy Loss
loss_function = torch.nn.CrossEntropyLoss()
# Explanation:
# - CrossEntropyLoss is a commonly used loss function for classification tasks.
# - It combines a softmax operation with a negative log-likelihood loss, making it ideal for multi-class or binary classification.

# 3. Optimizer: AdamW Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr)
# Explanation:
# - AdamW: A variant of the Adam optimizer with weight decay (regularization) to prevent overfitting.
# - `model.parameters()`: The model's learnable parameters to optimize.
# - `lr`: The learning rate specified earlier (1e-3).

# 4. Learning Rate Scheduler: Cosine Annealing
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
# Explanation:
# - CosineAnnealingLR adjusts the learning rate following a cosine curve.
# - `T_max=epochs`: The maximum number of epochs, ensuring the scheduler completes one full cycle.
# - Benefit: Starts with a high learning rate and gradually reduces it, helping the model converge smoothly.


In [ ]:
# =========================================
# Calculating the Total and Trainable Parameters of the Model
# =========================================

# Calculate the total number of parameters in the model
total_params = sum(param.numel() for param in model.parameters())
# Explanation:
# - `model.parameters()`: Retrieves all the parameters of the model.
# - `param.numel()`: Counts the total number of elements (parameters) in each tensor.
# - `sum()`: Adds up all the parameters to get the total count.

# Calculate the number of trainable parameters (parameters requiring gradients)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# Explanation:
# - `if p.requires_grad`: Filters parameters that require gradients (trainable during backpropagation).
# - Non-trainable parameters (e.g., frozen layers) are excluded.

# Print the total and trainable parameters
print(total_params, trainable_params)

# =========================================
# Expected Outcome:
# =========================================
# - Output: 11266626 11266626
#   - `11266626`: Total number of parameters in DenseNet121.
#   - `11266626`: Trainable parameters, indicating all layers are trainable.


**Metrics**

In [ ]:
# =========================================
# Defining Evaluation Metrics for the Model
# =========================================

# Comprehensive list of metrics for evaluating classification performance
all_metrics = [
    "sensitivity",                 # True Positive Rate (TPR): Proportion of actual positives correctly identified
    "specificity",                 # True Negative Rate (TNR): Proportion of actual negatives correctly identified
    "precision",                   # Positive Predictive Value (PPV): Proportion of predicted positives that are correct
    "negative predictive value",   # NPV: Proportion of predicted negatives that are correct
    "miss rate",                   # False Negative Rate (FNR): Proportion of actual positives missed
    "fall out",                    # False Positive Rate (FPR): Proportion of actual negatives incorrectly classified
    "false discovery rate",        # FDR: Proportion of predicted positives that are incorrect
    "false omission rate",         # FOR: Proportion of predicted negatives that are incorrect
    "prevalence threshold",        # Threshold where PPV = NPV given prevalence
    "threat score",                # Intersection over Union (IoU) for positives
    "accuracy",                    # Overall proportion of correct predictions
    "balanced accuracy",           # Average of sensitivity and specificity (useful for imbalanced datasets)
    "f1",                          # Harmonic mean of precision and recall
    "matthews correlation coefficient",  # MCC: Correlation between predicted and actual values (-1 to 1)
    "fowlkes mallows index",       # Geometric mean of precision and recall
    "informedness",                # Bookmaker Informedness (sensitivity + specificity - 1)
    "markedness"                   # Proportion of correct predictions based on prediction confidence
]

# Subset of common metrics from sklearn's classification report
sklearn_classification_report_metrics = [
    "precision",  # Positive Predictive Value: How many predicted positives are correct
    "recall",     # Sensitivity or True Positive Rate: Correctly identified positives
    "f1",         # F1-Score: Balance between precision and recall
    "accuracy"    # Overall classification accuracy
]


In [ ]:
# =========================================
# MONAI Metrics Function for Logging and Evaluation
# =========================================

# Function to compute and log performance metrics, save the model, and track the best metrics
def monai_metrics(y_true, y_predictions, metrics_list, split, epoch, best_metrics):
    """
    Computes AUROCC and confusion matrix metrics, logs them to WandB,
    and saves the model based on the best performance for specific metrics.

    Args:
        y_true (torch.Tensor): Ground truth labels.
        y_predictions (torch.Tensor): Model's raw output predictions (logits).
        metrics_list (list): List of metrics to compute using the confusion matrix.
        split (str): 'train' or 'validation' - indicates the dataset split.
        epoch (int): Current training epoch.
        best_metrics (dict): Dictionary to store the best metrics and corresponding epochs.

    Returns:
        best_metrics (dict): Updated dictionary with the best metrics and epochs.
    """

    # 1. Define MONAI Metrics
    aurocc_metric = ROCAUCMetric()
    confusion_matrix_metric = ConfusionMatrixMetric(
        metric_name=metrics_list,  # List of confusion matrix metrics to calculate
        reduction="none",          # No reduction across batches to calculate per-batch metrics
        include_background=True,   # Includes background class (if applicable)
        get_not_nans=False         # Excludes invalid (NaN) values
    )

    # Metrics to track for saving model checkpoints
    metrics_to_track = [
        "balanced accuracy",
        "f1",
        "matthews correlation coefficient"
    ]

    # 2. Postprocessing: Transform labels and predictions for metrics computation
    post_labels = Compose([AsDiscrete(to_onehot=2)])  # Convert labels to one-hot encoding (2 classes)
    post_predictions_aurocc = Compose([Activations(softmax=True)])  # Apply softmax to get probabilities
    post_predictions_confusionmatrix = Compose([AsDiscrete(argmax=True, to_onehot=2)])  # Get predicted classes

    # Apply transformations to the labels and predictions
    y_onehot = [post_labels(i) for i in decollate_batch(y_true, detach=False)]
    y_pred_aurocc = [post_predictions_aurocc(i) for i in decollate_batch(y_predictions, detach=False)]
    y_pred_confusionmatrix = [post_predictions_confusionmatrix(i) for i in decollate_batch(y_predictions, detach=False)]

    # 3. Compute AUROCC Metric
    aurocc_metric(y_pred_aurocc, y_onehot)  # Calculate AUROCC
    aurocc_result = aurocc_metric.aggregate()  # Aggregate the results
    print(f"{split.capitalize()} AUROCC: {aurocc_result:.4f}")

    # Log AUROCC to WandB
    wandb.log({f"{split.capitalize()} AUROCC": aurocc_result}, step=global_step)

    # Save model if AUROCC is the best so far (validation split only)
    if split == "validation":
        if "AUROCC" not in best_metrics or aurocc_result > best_metrics["AUROCC"]["value"]:
            best_metrics["AUROCC"] = {"value": aurocc_result, "epoch": epoch}
            torch.save(model.state_dict(), f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_best_AUROCC_fold{fold_number}.pth')

    # 4. Compute Confusion Matrix Metrics
    confusion_matrix_metric(y_pred_confusionmatrix, y_onehot)
    confusion_matrix_results = confusion_matrix_metric.aggregate(reduction="mean_batch")

    metrics_dict = {}

    # Loop through the list of metrics and calculate values
    for i, name in enumerate(metrics_list):
        mean_metric_class_zero = confusion_matrix_results[i][0].item()  # Metric for class 0
        mean_metric_class_one = confusion_matrix_results[i][1].item()  # Metric for class 1

        if name == "accuracy":
            # Report accuracy as a percentage
            metric_value = mean_metric_class_zero * 100
            print(f"{split.capitalize()} {name.capitalize()}: {metric_value:.2f}%")
        else:
            # Report other metrics as the average for class 0 and class 1
            metric_value = (mean_metric_class_zero + mean_metric_class_one) / 2
            print(f"{split.capitalize()} {name.capitalize()}: {metric_value:.4f}")

        # Add the metric to the dictionary for logging
        metrics_dict[f"{split.capitalize()} {name.capitalize()}"] = metric_value

    # Log confusion matrix metrics to WandB
    wandb.log(metrics_dict, step=global_step)

    # 5. Save Model Based on Best Confusion Matrix Metrics (Validation Only)
    if split == "validation":
        for i, name in enumerate(metrics_list):
            mean_metric_class_zero = confusion_matrix_results[i][0].item()
            mean_metric_class_one = confusion_matrix_results[i][1].item()
            mean_metric = (mean_metric_class_zero + mean_metric_class_one) / 2

            if name == "accuracy":
                mean_metric *= 100  # Convert accuracy to percentage

            # Update best metrics dictionary and save model checkpoint
            if name not in best_metrics or mean_metric > best_metrics[name]["value"]:
                best_metrics[name] = {"value": mean_metric, "epoch": epoch}
                if name in metrics_to_track:
                    name = name.replace(' ', '_')  # Replace spaces with underscores for filenames
                    torch.save(model.state_dict(), f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_best_{name}_fold{fold_number}.pth')

    # 6. Reset Metrics for the Next Round
    aurocc_metric.reset()
    confusion_matrix_metric.reset()

    # Clear memory for transformed predictions
    del y_onehot, y_pred_aurocc, y_pred_confusionmatrix

    # Return updated best metrics (validation only)
    if split == "validation":
        return best_metrics


**Training**

We used W&B for logging our metrics and monitoring the training.

In [ ]:
# =========================================
# Sanity Check: Confirm Results Directory
# =========================================

# Print a message to confirm where the results will be stored
print(f'ATTENTION: results will be stored in {results_folder}')


In [ ]:
# =========================================
# W&B Initialization and Training Loop
# =========================================

# Naming convention for MRI sequences used for training
mrisq = [mri.split('_')[0] for mri in mri_to_use]  # Extract MRI names (e.g., 'CT1', 'T2') without suffix
mri_sequences = '_'.join(mrisq)  # Join MRI names into a single string

# Initialize a Weights & Biases (W&B) project to log metrics and configurations
wandb.init(project='Choose_your_project_ID_name')

# =========================================
# W&B Configuration
# =========================================
# Log configuration settings for better experiment tracking
config = wandb.config
config.preprocessing = preprocessing_mode               # Preprocessing mode (e.g., 'FETS-std')
config.MRI_Sequences = mri_sequences                    # MRI sequences used (e.g., 'CT1_T2')
config.learning_rate = lr                               # Learning rate
config.batch_size = bs                                  # Batch size
config.mode = '3D'                                      # Model input mode (3D data)
config.backbone = nn_architecture                       # Model backbone architecture (e.g., DenseNet121)
config.total_parameters = total_params                  # Total model parameters
config.trainable_parameters = trainable_params          # Trainable model parameters
config.optimizer = 'AdamW'                              # Optimizer used for training
config.normalization = 'Per patient'                    # Normalization method
config.epochs = epochs                                  # Total number of training epochs
config.augmentation = 'Affine'                          # Data augmentation applied
config.RandAffine_tran_scale_rotate_prob = [(15,15,10),(0.05,0.05,0.05),(math.pi/8,math.pi/8,math.pi/8),0.5]
config.RandGaussianNoise_prob_mean_std = [0.5, 0.0, 0.2]
config.RandFlip_prob_axis = [0.5, (0,1,2)]

# Conditional config for Spatial Padding
if preprocessing_mode == 'FETS-std':
    config.SpatialPad = 'NA'
else:
    config.SpatialPad = (168, 196, 168)

# Set a unique name for the W&B run
wandb.run.name = f'name_of_your_choice_for_your_run_{mri_sequences}'

# Initialize metrics tracking and global step counter
best_metrics = {}  # Dictionary to store the best performance metrics
training_results = []  # List to store results for each epoch
global_step = 0  # Counter for logging steps to W&B

# =========================================
# Training and Validation Loop
# =========================================
for i, epoch in enumerate(tqdm(range(epochs))):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{epochs}")
    model.train()

    epoch_loss = 0
    step = 0
    step_loss_list = []
    y_pred = torch.tensor([], dtype=torch.float32, device=device)
    y = torch.tensor([], dtype=torch.long, device=device)

    # Training Loop
    for batch_data in train_loader:
        step += 1
        global_step += 1

        # Move input and labels to the specified device (GPU/CPU)
        inputs, labels = batch_data['concat_mri'].to(device), batch_data['label_column_name'].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)  # Compute loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        # Log step loss to W&B
        wandb.log({"Training step loss": loss.item()}, step=global_step)
        step_loss_list.append(loss.item())

        # Accumulate predictions and labels for metrics computation
        y = torch.cat([y, labels], dim=0)
        y_pred = torch.cat([y_pred, outputs], dim=0)

    # Learning Rate Scheduler Step
    wandb.log({'lr': lr_scheduler.get_lr()[0]}, step=global_step)
    lr_scheduler.step()

    # Log epoch-level metrics
    median_epoch_loss = statistics.median(step_loss_list)
    wandb.log({'Training median epoch loss': median_epoch_loss}, step=global_step)
    epoch_loss /= step
    wandb.log({'Training epoch loss': epoch_loss}, step=global_step)
    print(f"Training epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # Compute and log metrics using MONAI metrics function
    best_metrics_dict = monai_metrics(y, y_pred, all_metrics, epoch, best_metrics)

    # Store predictions, probabilities, and labels
    y_dict = y.detach().cpu().numpy()
    y_pred_classes = y_pred.detach().cpu().argmax(dim=1).numpy()
    y_pred_prob = torch.softmax(y_pred.detach().cpu(), dim=1).numpy()
    training_results_dict = {
        "epoch": epoch,
        "predictions": y_pred_classes.tolist(),
        "probabilities": y_pred_prob.tolist(),
        "labels": y_dict.tolist()
    }
    training_results.append(training_results_dict)

    # Clean up predictions and labels to free memory
    del y, y_pred

    # Save the model checkpoint for the current epoch
    torch.save(model.state_dict(), f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_epoch{epoch}.pth')

# Finalize W&B run
wandb.finish()

# Save the final model after all epochs
torch.save(model.state_dict(), f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_last_epoch.pth')

# Save training results to a JSON file for all epochs
with open(f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_training_results_.json', "w") as jsonfile:
    json.dump(training_results, jsonfile)

# Save the best metrics to a CSV file
df_best_metrics = pd.DataFrame.from_dict(best_metrics_dict, orient="index")
df_best_metrics.index.name = "Metric"
df_best_metrics.to_csv((f'{results_folder}/{nn_architecture}_{preprocessing_mode}_{mri_sequences}_best_metrics.csv'))

print('ALL DONE!')
